# 🧭 验证 ui_present_proposal → action_candidate

**目的：** 验证 agent 在注入 `ui_present_proposal` 前端工具后，能否正确返回 `action_candidate`。

**背景问题：** 真实请求中 `tools[]` 有 `ui_present_proposal`，但 agent 读的是
`state['copilotkit']['actions']`。若 state 没注入这个字段，LLM 看不到该工具，不会调用。

**验证矩阵：**

| 场景 | 用户消息 | 期望行为 |
|------|----------|----------|
| A | 当前有哪些项目 | 调用 `list_projects` 返回数据（非导航） |
| B | 帮我去数据模型管理 | 调用 `ui_present_proposal` 返回 `action_candidate` |
| C | 帮我配置权限 | 调用 `ui_present_proposal` 返回 `action_candidate` |
| D | 项目 | 意图模糊，返回 `clarification_candidate` |


## 0. 环境初始化

In [2]:
import sys, os, json, uuid
from unittest.mock import AsyncMock, patch, MagicMock

AGENT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if AGENT_ROOT not in sys.path:
    sys.path.insert(0, AGENT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(AGENT_ROOT, '.env'))

from logging_setup import setup_logging
setup_logging()

import config
print('AGENT_ROOT:', AGENT_ROOT)
print('LLM_MODEL :', config.LLM_MODEL)

AGENT_ROOT: /data/home/lukemxjia/modelcraft/modelcraft-agent
LLM_MODEL : deepseek-chat


## 1. 前端工具定义（模拟 CopilotKit 注入的 actions）

`agent_node` 从 `state['copilotkit']['actions']` 读取前端工具，
这里按真实请求 payload 原样构造。

In [3]:
# 与真实请求 body.tools[] 保持一致
SHOW_TOAST_TOOL = {
    "name": "show_toast",
    "description": "向用户显示一条临时通知消息（不需要用户在聊天框内查看）",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {"type": "string", "description": "通知内容"},
            "type": {"type": "string", "description": "success | error | info | warning（默认 info）"}
        },
        "required": ["message"]
    }
}

SHOW_NAVIGATION_PROPOSAL_TOOL = {
    "name": "ui_present_proposal",
    "description": "向用户展示 AI 导航提案卡片，用户可点击候选项执行页面跳转、元素高亮或发送澄清消息",
    "parameters": {
        "type": "object",
        "properties": {
            "response": {
                "type": "object",
                "properties": {}
            }
        },
        "required": ["response"]
    }
}

# 真实请求中的 routeCatalog context
ROUTE_CATALOG = [
    {"routeTemplate": "/org/:orgName/workspace", "title": "项目列表", "description": "查看、搜索、创建和管理组织下的所有项目", "keywords": ["项目列表", "所有项目", "workspace"]},
    {"routeTemplate": "/org/:orgName/project/:projectSlug/model-editor", "title": "数据模型编辑器", "description": "创建和管理数据模型、字段结构", "keywords": ["模型", "字段", "数据模型", "模型编辑器"]},
    {"routeTemplate": "/org/:orgName/project/:projectSlug/rbac/roles", "title": "RBAC 角色管理", "description": "管理项目内的角色与权限包", "keywords": ["权限", "RBAC", "角色"]},
    {"routeTemplate": "/org/:orgName/project/:projectSlug/end-users", "title": "终端用户管理", "description": "管理访问本项目的终端用户账号", "keywords": ["终端用户", "end user"]},
    {"routeTemplate": "/org/:orgName/project/:projectSlug/settings", "title": "项目设置", "description": "修改项目基本信息、管理数据库集群", "keywords": ["项目设置", "集群配置", "数据库连接"]},
]

AI_TARGETS = [
    {"id": "create_project", "label": "新建项目", "description": "点击打开新建项目表单"}
]

print('✅ 前端工具定义完成')
print(f'  - show_toast')
print(f'  - ui_present_proposal')
print(f'  routeCatalog: {len(ROUTE_CATALOG)} 条')
print(f'  aiTargets: {len(AI_TARGETS)} 条')

✅ 前端工具定义完成
  - show_toast
  - ui_present_proposal
  routeCatalog: 5 条
  aiTargets: 1 条


## 2. Mock GraphQL Client

In [4]:
FAKE_PROJECTS = [
    {"id": "p1", "slug": "onboardceshi", "title": "onboard测试", "description": "", "status": "ACTIVE"},
    {"id": "p2", "slug": "abcde",        "title": "abcde",       "description": "", "status": "ACTIVE"},
    {"id": "p3", "slug": "pro1",         "title": "pro1",        "description": "", "status": "ACTIVE"},
]

def make_fake_client(state=None):
    c = MagicMock()
    c.list_projects    = AsyncMock(return_value={"data": {"projects": FAKE_PROJECTS}})
    c.list_databases   = AsyncMock(return_value={"data": {"listDatabases": {"edges": []}}}  )
    c.list_models      = AsyncMock(return_value={"data": {"models": {"items": [], "hasNextPage": False}}})
    c.get_model_fields = AsyncMock(return_value={"data": {"model": {"fields": []}}})
    c.query_model      = AsyncMock(return_value={"data": {"queryModel": {"items": [], "hasNextPage": False}}})
    c.nl2filter        = AsyncMock(return_value={"data": {"nl2filter": {"filter": "{}", "explanation": "mock"}}})
    return c

# ⚠️  tools.py 做的是 `from agents.shared import make_client`，
#    所以必须 patch agents.tools.make_client（本地引用），
#    而非 agents.shared.make_client（patch 了也不影响已经绑定的名字）
PATCH_TARGET = 'agents.tools.make_client'
print(f'✅ mock 就绪，patch target = {PATCH_TARGET}')

✅ mock 就绪，patch target = agents.tools.make_client


## 3. State 构造函数

**关键：** 注入 `copilotkit.actions`，让 agent_node 能把前端工具绑到 LLM。

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage

ORG_NAME     = "lukeco"
PROJECT_SLUG = "onboardceshi"
LAYER        = "org"

# routeCatalog 注入为 SystemMessage（模拟 CopilotKit useCopilotReadable 的效果）
ROUTE_CATALOG_MSG = SystemMessage(content=(
    "系统所有可导航页面目录（routeCatalog）。"
    "调用 ui_present_proposal 时，ui.navigate 的 route 字段必须从 routeTemplate 派生，"
    "将 :orgName 替换为实际 orgName，:projectSlug 替换为实际 projectSlug。\n"
    + json.dumps(ROUTE_CATALOG, ensure_ascii=False)
))

AI_TARGETS_MSG = SystemMessage(content=(
    "当前页面已注册的 AI 高亮目标（aiTargets）。"
    "调用 ui_present_proposal 时，ui.highlight 的 targetId 必须从这个列表中选取。\n"
    + json.dumps(AI_TARGETS, ensure_ascii=False)
))

def make_state(message: str, inject_frontend_tools: bool = True) -> dict:
    """
    inject_frontend_tools=True 时模拟真实 CopilotKit 请求（含 ui_present_proposal）。
    routeCatalog 和 aiTargets 作为 SystemMessage 注入 messages，模拟 useCopilotReadable 效果。
    """
    state: dict = {
        "messages"     : [ROUTE_CATALOG_MSG, AI_TARGETS_MSG, HumanMessage(content=message)],
        "authorization": "Bearer fake-offline-token",
        "org_name"     : ORG_NAME,
        "layer"        : LAYER,
        "project_slug" : PROJECT_SLUG,
        "current_model": "",
        "current_db"   : "",
    }
    if inject_frontend_tools:
        state["copilotkit"] = {
            "actions": [SHOW_TOAST_TOOL, SHOW_NAVIGATION_PROPOSAL_TOOL],
        }
    return state

def new_cfg() -> dict:
    tid = str(uuid.uuid4())
    return {"configurable": {"thread_id": tid}}

# 每次测试前重置 LazyGraph，让 graph 在 mock 上下文内重新编译
def reset_graph():
    from agents import admin_agent
    admin_agent._LazyGraph._instance = None

print('✅ make_state 已定义（routeCatalog + aiTargets 注入为 SystemMessage）')
print('✅ reset_graph 已定义（每次测试前调用，确保 mock 生效）')

✅ make_state 已定义（routeCatalog + aiTargets 注入为 SystemMessage）
✅ reset_graph 已定义（每次测试前调用，确保 mock 生效）


## 4. 工具函数：打印 tool_calls 详情

In [6]:
from langchain_core.messages import AIMessage, ToolMessage

def _parse_response(raw) -> dict:
    """response 可能是 dict 或 JSON 字符串，统一转成 dict。"""
    if isinstance(raw, dict):
        return raw
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except Exception:
            return {"_raw": raw}
    return {}

def inspect_result(result: dict, label: str = ""):
    """打印消息链，重点展示 tool_calls 和 ui_present_proposal 的 response 参数。"""
    msgs = result["messages"]
    print(f"\n{'='*60}")
    if label:
        print(f"  {label}")
    print(f"  共 {len(msgs)} 条消息")
    print('='*60)

    for i, m in enumerate(msgs):
        role = type(m).__name__
        if isinstance(m, AIMessage) and getattr(m, 'tool_calls', None):
            for tc in m.tool_calls:
                name = tc['name']
                args = tc.get('args', {})
                if name == 'ui_present_proposal':
                    raw_resp = args.get('response', {})
                    print(f"  [{i}] AIMessage → 🧭 ui_present_proposal")
                    print(f"       response type: {type(raw_resp).__name__}  ← {'✅ dict' if isinstance(raw_resp, dict) else '⚠️  string (LLM 序列化问题)'}")
                    resp = _parse_response(raw_resp)
                    candidates = resp.get('candidates', [])
                    print(f"       message     : {resp.get('message', '')}")
                    print(f"       proposalType: {resp.get('proposalType', '')}")
                    print(f"       candidates ({len(candidates)}):")
                    for c in candidates:
                        ctype = c.get('type', '?')
                        ctitle = c.get('title', '')
                        actions = c.get('actions', [])
                        print(f"         {'✅' if ctype == 'action_candidate' else '❓'} [{ctype}] {ctitle}")
                        for a in actions:
                            print(f"              {a.get('type')} → {json.dumps(a.get('args', {}), ensure_ascii=False)}")
                else:
                    print(f"  [{i}] AIMessage → 🔧 {name}({json.dumps(args, ensure_ascii=False)[:80]})")
        elif isinstance(m, ToolMessage):
            print(f"  [{i}] ToolMessage  → {str(m.content)[:80]}")
        elif isinstance(m, AIMessage):
            print(f"  [{i}] AIMessage  text={str(m.content)[:80]}")
        else:
            print(f"  [{i}] {role}  {str(m.content)[:80]}")

print('✅ inspect_result 已定义（支持 response 为 string 的情况）')

✅ inspect_result 已定义（支持 response 为 string 的情况）


## 5. 场景 A：数据查询「当前有哪些项目」

**期望：** 调用 `list_projects` 返回数据，不调用 `ui_present_proposal`

In [7]:
from agents.admin_agent import admin_graph

reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    result_a = await admin_graph.ainvoke(
        make_state("当前有哪些项目"),
        config=new_cfg()
    )

inspect_result(result_a, "场景 A：当前有哪些项目")

{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "ui_present_proposal"], "latest_user_text": "\u5f53\u524d\u6709\u54ea\u4e9b\u9879\u76ee", "is_direct_nav_intent": false, "is_list_nav_intent": true, "history_has_list_tools": false, "history_has_proposal": false, "force_proposal_on_this_turn": false, "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-26T11:24:48.592241Z"}
{"service": "modelcraft-agent", "tool_name": "list_projects", "args_summary": "", "event": "tool.call.start", "level": "info", "timestamp": "2026-05-26T11:24:49.488919Z"}
{"service": "modelcraft-agent", "tool_name": "list_projects", "duration_ms": 3.66, "success": true, "event": "tool.call.end", "level": "info", "timestamp": "2026-05-26T11:24:49.493021Z"}
{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "ui_present_proposal"], "latest_user_

## 6. 场景 B：导航请求「帮我去数据模型管理」

**期望：** 调用 `ui_present_proposal`，返回 `action_candidate`，
route 为 `/org/lukeco/project/onboardceshi/model-editor`

In [8]:
reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    result_b = await admin_graph.ainvoke(
        make_state("帮我去数据模型管理"),
        config=new_cfg()
    )

inspect_result(result_b, "场景 B：帮我去数据模型管理")

{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "ui_present_proposal"], "latest_user_text": "\u5e2e\u6211\u53bb\u6570\u636e\u6a21\u578b\u7ba1\u7406", "is_direct_nav_intent": true, "is_list_nav_intent": false, "history_has_list_tools": false, "history_has_proposal": false, "force_proposal_on_this_turn": true, "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-26T11:24:51.879969Z"}



  场景 B：帮我去数据模型管理
  共 4 条消息
  [0] SystemMessage  系统所有可导航页面目录（routeCatalog）。调用 ui_present_proposal 时，ui.navigate 的 route 字段必须从 rou
  [1] SystemMessage  当前页面已注册的 AI 高亮目标（aiTargets）。调用 ui_present_proposal 时，ui.highlight 的 targetId 必须从
  [2] HumanMessage  帮我去数据模型管理
  [3] AIMessage → 🧭 ui_present_proposal
       response type: dict  ← ✅ dict
       message     : 正在跳转到数据模型编辑器...
       proposalType: navigation
       candidates (1):
         ✅ [action_candidate] 数据模型编辑器
              ui.navigate → {"route": "/org/lukeco/project/onboardceshi/model-editor"}


## 7. 场景 C：导航请求「帮我配置权限」

**期望：** 调用 `ui_present_proposal`，返回 `action_candidate`，
route 包含 `/rbac/roles`

In [9]:
reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    result_c = await admin_graph.ainvoke(
        make_state("帮我配置权限"),
        config=new_cfg()
    )

inspect_result(result_c, "场景 C：帮我配置权限")

{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "ui_present_proposal"], "latest_user_text": "\u5e2e\u6211\u914d\u7f6e\u6743\u9650", "is_direct_nav_intent": true, "is_list_nav_intent": false, "history_has_list_tools": false, "history_has_proposal": false, "force_proposal_on_this_turn": true, "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-26T11:24:53.257566Z"}



  场景 C：帮我配置权限
  共 4 条消息
  [0] SystemMessage  系统所有可导航页面目录（routeCatalog）。调用 ui_present_proposal 时，ui.navigate 的 route 字段必须从 rou
  [1] SystemMessage  当前页面已注册的 AI 高亮目标（aiTargets）。调用 ui_present_proposal 时，ui.highlight 的 targetId 必须从
  [2] HumanMessage  帮我配置权限
  [3] AIMessage → 🧭 ui_present_proposal
       response type: dict  ← ✅ dict
       message     : 好的，我来帮您导航到权限配置页面。当前组织为 **lukeco**，项目为 **onboardceshi**。
       proposalType: navigation
       candidates (2):
         ✅ [action_candidate] RBAC 角色管理
              ui.navigate → {"route": "/org/lukeco/project/onboardceshi/rbac/roles"}
         ✅ [action_candidate] 终端用户管理
              ui.navigate → {"route": "/org/lukeco/project/onboardceshi/end-users"}


## 8. 场景 D：模糊意图「项目」

**期望：** 返回 `clarification_candidate`（意图不明），或 `action_candidate`（直接跳项目列表）

In [10]:
reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    result_d = await admin_graph.ainvoke(
        make_state("项目"),
        config=new_cfg()
    )

inspect_result(result_d, "场景 D：模糊意图 '项目'")

{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "ui_present_proposal"], "latest_user_text": "\u9879\u76ee", "is_direct_nav_intent": false, "is_list_nav_intent": false, "history_has_list_tools": false, "history_has_proposal": false, "force_proposal_on_this_turn": false, "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-26T11:24:55.181040Z"}


{"service": "modelcraft-agent", "tool_name": "list_projects", "args_summary": "", "event": "tool.call.start", "level": "info", "timestamp": "2026-05-26T11:24:56.293345Z"}
{"service": "modelcraft-agent", "tool_name": "list_projects", "duration_ms": 3.5, "success": true, "event": "tool.call.end", "level": "info", "timestamp": "2026-05-26T11:24:56.297309Z"}
{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "ui_present_proposal"], "latest_user_text": "\u9879\u76ee", "is_direct_nav_intent": false, "is_list_nav_intent": false, "history_has_list_tools": true, "history_has_proposal": false, "force_proposal_on_this_turn": false, "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-26T11:24:56.298560Z"}

  场景 D：模糊意图 '项目'
  共 6 条消息
  [0] SystemMessage  系统所有可导航页面目录（routeCatalog）。调用 ui_present_proposal 时，ui.navigate 的 route 字段必须从 rou
  [1] SystemMessage  当前页面已注册的 AI 高亮目标（aiTargets）。调用 ui_present_proposa

## 9. 对比：不注入前端工具时的行为

验证 **根本原因**：没有 `copilotkit.actions` 时 LLM 看不到 `ui_present_proposal`，只会用文字描述

In [11]:
reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    result_no_tools = await admin_graph.ainvoke(
        make_state("帮我去数据模型管理", inject_frontend_tools=False),
        config=new_cfg()
    )

print("\n[不注入前端工具] Agent 回复：")
print(result_no_tools['messages'][-1].content)
print("\n↑ 只有文字描述，没有 ui_present_proposal 调用 ← 这就是 bug 的根本原因")

{"service": "modelcraft-agent", "copilotkit_keys": [], "frontend_action_count": 0, "frontend_action_names": [], "latest_user_text": "\u5e2e\u6211\u53bb\u6570\u636e\u6a21\u578b\u7ba1\u7406", "is_direct_nav_intent": true, "is_list_nav_intent": false, "history_has_list_tools": false, "history_has_proposal": false, "force_proposal_on_this_turn": false, "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-26T11:24:58.629446Z"}


{"service": "modelcraft-agent", "tool_name": "list_projects", "args_summary": "", "event": "tool.call.start", "level": "info", "timestamp": "2026-05-26T11:24:59.701754Z"}
{"service": "modelcraft-agent", "tool_name": "list_projects", "duration_ms": 3.35, "success": true, "event": "tool.call.end", "level": "info", "timestamp": "2026-05-26T11:24:59.705549Z"}
{"service": "modelcraft-agent", "copilotkit_keys": [], "frontend_action_count": 0, "frontend_action_names": [], "latest_user_text": "\u5e2e\u6211\u53bb\u6570\u636e\u6a21\u578b\u7ba1\u7406", "is_direct_nav_intent": true, "is_list_nav_intent": false, "history_has_list_tools": true, "history_has_proposal": false, "force_proposal_on_this_turn": false, "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-26T11:24:59.706933Z"}
{"service": "modelcraft-agent", "copilotkit_keys": [], "frontend_action_count": 0, "frontend_action_names": [], "latest_user_text": "\u5e2e\u6211\u53bb\u6570\u636e\u6a21\u578b\u7ba1\u7406", "is_direct_n

## 10. 断言验证（自动检查）

In [12]:
def get_tool_calls(result: dict) -> list[dict]:
    """提取所有 AIMessage 中的 tool_calls。"""
    calls = []
    for m in result['messages']:
        if isinstance(m, AIMessage) and getattr(m, 'tool_calls', None):
            calls.extend(m.tool_calls)
    return calls

def get_proposal_candidates(result: dict) -> list[dict]:
    """提取 ui_present_proposal 调用中的 candidates（自动处理 string/dict）。"""
    for tc in get_tool_calls(result):
        if tc['name'] == 'ui_present_proposal':
            resp = _parse_response(tc.get('args', {}).get('response', {}))
            return resp.get('candidates', [])
    return []

passed = []
failed = []

def check(name, cond, detail=""):
    if cond:
        passed.append(name)
        print(f"  ✅ {name}")
    else:
        failed.append(name)
        print(f"  ❌ {name}  {detail}")

print("\n── 断言结果 ──────────────────────────────────────")

# A: 数据查询 → 调用 list_projects，不调用 ui_present_proposal
calls_a = [tc['name'] for tc in get_tool_calls(result_a)]
check("A: list_projects 被调用", "list_projects" in calls_a, f"实际: {calls_a}")
check("A: 未调用 ui_present_proposal", "ui_present_proposal" not in calls_a, f"实际: {calls_a}")

# B: 导航 → response 是 dict，含 action_candidate，route 含 model-editor
cands_b = get_proposal_candidates(result_b)
action_cands_b = [c for c in cands_b if c.get('type') == 'action_candidate']
check("B: ui_present_proposal 被调用", len(cands_b) > 0, "candidates 为空")
check("B: 含 action_candidate", len(action_cands_b) > 0, f"实际类型: {[c.get('type') for c in cands_b]}")
# response type check
b_tool_calls = [tc for tc in get_tool_calls(result_b) if tc['name'] == 'ui_present_proposal']
if b_tool_calls:
    raw = b_tool_calls[0].get('args', {}).get('response', {})
    check("B: response 是 dict（非字符串）", isinstance(raw, dict), f"实际类型: {type(raw).__name__}")
if action_cands_b:
    routes = [a.get('args', {}).get('route', '') for c in action_cands_b for a in c.get('actions', []) if a.get('type') == 'ui.navigate']
    check("B: route 包含 model-editor", any('model-editor' in r for r in routes), f"实际 routes: {routes}")

# C: 权限导航 → action_candidate，route 含 rbac
cands_c = get_proposal_candidates(result_c)
action_cands_c = [c for c in cands_c if c.get('type') == 'action_candidate']
check("C: ui_present_proposal 被调用", len(cands_c) > 0, "candidates 为空")
check("C: 含 action_candidate", len(action_cands_c) > 0, f"实际类型: {[c.get('type') for c in cands_c]}")
if action_cands_c:
    routes_c = [a.get('args', {}).get('route', '') for c in action_cands_c for a in c.get('actions', []) if a.get('type') == 'ui.navigate']
    check("C: route 包含 rbac", any('rbac' in r for r in routes_c), f"实际 routes: {routes_c}")

# D: 模糊意图「项目」— 接受两种合理行为：
#    - ui_present_proposal（导航候选）
#    - list_projects（数据查询，也合理）
calls_d = [tc['name'] for tc in get_tool_calls(result_d)]
cands_d = get_proposal_candidates(result_d)
nav_called = len(cands_d) > 0
data_called = "list_projects" in calls_d
check(
    "D: 导航 or 数据查询（两种都合理）",
    nav_called or data_called,
    f"既没有 ui_present_proposal 也没有 list_projects，tool_calls={calls_d}"
)
if nav_called:
    print(f"       → 选择了导航路径（ui_present_proposal）")
elif data_called:
    print(f"       → 选择了数据查询路径（list_projects），可接受")

print(f"\n{'='*50}")
print(f"  通过: {len(passed)} / 失败: {len(failed)}")
if failed:
    print(f"  ❌ 失败项: {failed}")
else:
    print("  🎉 全部通过")


── 断言结果 ──────────────────────────────────────
  ✅ A: list_projects 被调用
  ❌ A: 未调用 ui_present_proposal  实际: ['list_projects', 'ui_present_proposal']
  ✅ B: ui_present_proposal 被调用
  ✅ B: 含 action_candidate
  ✅ B: response 是 dict（非字符串）
  ✅ B: route 包含 model-editor
  ✅ C: ui_present_proposal 被调用
  ✅ C: 含 action_candidate
  ✅ C: route 包含 rbac
  ✅ D: 导航 or 数据查询（两种都合理）
       → 选择了导航路径（ui_present_proposal）

  通过: 9 / 失败: 1
  ❌ 失败项: ['A: 未调用 ui_present_proposal']


## 11. 场景 E：导航循环回归测试

**问题背景：** CopilotKit ag-ui 协议在前端工具渲染后自动触发新一轮 run，
新 run 的 `_latest_user_text` 仍然是原始用户消息（如"帮我导航到访问控制"），
导致 `_should_force_proposal_on_turn` 每次都返回 True → 无限循环。

**修复：** `_should_force_proposal_on_turn` 新增 `history_has_proposal` 参数，
当消息历史中已有 `ui_present_proposal` tool call 时，不再强制调用。

**本场景模拟：** 同一个 thread 连续 3 轮 ainvoke（模拟 CopilotKit 反复触发），
验证第 2、3 轮不会再次调用 `ui_present_proposal`。

In [ ]:
# 场景 E：导航循环回归测试
# 模拟 CopilotKit 在前端工具渲染后自动重发请求，验证不会无限循环

reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    cfg = new_cfg()
    nav_msg = "帮我导航到访问控制"

    # 第 1 轮：正常导航请求
    result_e1 = await admin_graph.ainvoke(make_state(nav_msg), config=cfg)
    calls_e1 = [tc['name'] for tc in get_tool_calls(result_e1)]
    proposal_count_e1 = calls_e1.count('ui_present_proposal')
    print(f"第 1 轮 tool_calls: {calls_e1}")
    print(f"  ui_present_proposal 调用次数: {proposal_count_e1}")

    # 第 2 轮：模拟 CopilotKit 自动重发（同一个 thread_id）
    # 用第 1 轮的 state 继续调用，state 中已包含 ui_present_proposal 的 AIMessage
    result_e2 = await admin_graph.ainvoke(result_e1, config=cfg)
    calls_e2 = [tc['name'] for tc in get_tool_calls(result_e2)]
    # 只统计新增的 tool_calls（排除第 1 轮遗留的）
    prev_msg_count = len(result_e1['messages'])
    new_calls_e2 = []
    for m in result_e2['messages'][prev_msg_count:]:
        if isinstance(m, AIMessage) and getattr(m, 'tool_calls', None):
            new_calls_e2.extend(tc['name'] for tc in m.tool_calls)
    print(f"第 2 轮 新增 tool_calls: {new_calls_e2}")
    print(f"  是否又调用了 ui_present_proposal: {'YES - 循环未修复!' if 'ui_present_proposal' in new_calls_e2 else 'NO - 循环已修复'}")

    # 第 3 轮：再验证一次
    result_e3 = await admin_graph.ainvoke(result_e2, config=cfg)
    prev_msg_count2 = len(result_e2['messages'])
    new_calls_e3 = []
    for m in result_e3['messages'][prev_msg_count2:]:
        if isinstance(m, AIMessage) and getattr(m, 'tool_calls', None):
            new_calls_e3.extend(tc['name'] for tc in m.tool_calls)
    print(f"第 3 轮 新增 tool_calls: {new_calls_e3}")
    print(f"  是否又调用了 ui_present_proposal: {'YES - 循环未修复!' if 'ui_present_proposal' in new_calls_e3 else 'NO - 循环已修复'}")

# 断言
print("\n── 场景 E 断言 ──────────────────────────────────────")
check("E-1: 第 1 轮调用了 ui_present_proposal", proposal_count_e1 >= 1, f"实际: {proposal_count_e1}")
check("E-2: 第 2 轮未再调用 ui_present_proposal", 'ui_present_proposal' not in new_calls_e2, f"实际: {new_calls_e2}")
check("E-3: 第 3 轮未再调用 ui_present_proposal", 'ui_present_proposal' not in new_calls_e3, f"实际: {new_calls_e3}")

print(f"\n{'='*50}")
print(f"  场景 E 通过: {sum(1 for p in passed if p.startswith('E'))} / 失败: {sum(1 for f in failed if f.startswith('E'))}")

SyntaxError: unterminated string literal (detected at line 45) (4257759720.py, line 45)